# 04 — Feature ablation

Does the feature engineering earn its complexity? One XGBoost classifier with
default hyperparameters is trained four times on progressively larger feature
sets drawn from KNHANES, and the four are compared on a common held-out split.

Produces thesis Table `add features and interaction`.

| Set | Features |
|---|---|
| baseline | 9 routinely measured variables |
| baseline + interactions | 57 |
| extended | 17 measured variables |
| extended + interactions | 241 |

The `RACE = 1` feature table is the right input here. The legacy ablation
notebook called the interaction generator without a `RACE` argument and so took
its default, tagging Korean participants with the US code. That cannot affect
these results — `RACE` is constant within a single cohort and these runs use
default hyperparameters, so no tree can split on it and no column sampling
reaches it — and it is reproduced rather than corrected (`docs/audit.md` F23).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import polars as pl

from src.data.io import output_path, processed_path
from src.features.selection import ablation_feature_sets, drop_target_derived
from src.logging_utils import configure_logging
from src.models.evaluate import compute_metrics
from src.models.train import split_frames, train_classifier

configure_logging(ROOT / "logs")

features = drop_target_derived(
    pl.read_parquet(processed_path("KNHANES_race1_features.parquet"))
)
feature_sets = ablation_feature_sets(features)
print(f"after dropping identifiers and target-derived columns: {features.shape}")

2026-09-13 21:56:14 [INFO] src.features.selection: baseline: 9 features


2026-09-13 21:56:14 [INFO] src.features.selection: baseline + interactions: 57 features


2026-09-13 21:56:14 [INFO] src.features.selection: extended: 17 features


2026-09-13 21:56:14 [INFO] src.features.selection: extended + interactions: 241 features


after dropping identifiers and target-derived columns: (15138, 242)


## Train and evaluate

Each set is split 70/30 with stratification and a fixed seed, so all four models
are scored on the same participants. Class imbalance is handled by weighting —
`scale_pos_weight` — rather than by resampling.

In [2]:
runs = {}
rows = []
for name, frame in feature_sets.items():
    run = train_classifier(frame, algorithm="XGBoost")
    metrics = compute_metrics(run["y_test"], run["preds"])
    runs[name] = run
    rows.append(
        {
            "feature_set": name,
            "n_features": frame.width - 1,
            **{key: value for key, value in metrics.items() if key != "confusion_matrix"},
            **metrics["confusion_matrix"],
        }
    )

# Thesis ordering: baseline, baseline + interactions, extended, extended + interactions.
ablation = pl.DataFrame(rows).drop("optimal_threshold")
ablation.to_pandas().to_excel(output_path("feature_ablation.xlsx"), index=False)
ablation

2026-09-13 21:56:14 [INFO] src.models.train: XGBoost on 9 features: 10596 training / 4542 test rows


2026-09-13 21:56:14 [INFO] src.models.train: XGBoost on 57 features: 10596 training / 4542 test rows


2026-09-13 21:56:15 [INFO] src.models.train: XGBoost on 17 features: 10596 training / 4542 test rows


2026-09-13 21:56:15 [INFO] src.models.train: XGBoost on 241 features: 10596 training / 4542 test rows


feature_set,n_features,roc_auc,accuracy,sensitivity(recall),specificity,PPV(precision),NPV,f1_score,Youdens_Index,pr_auc,TP,TN,FP,FN
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64
"""baseline""",9,0.847,0.785,0.699,0.818,0.598,0.875,0.645,0.517,0.709,886,2679,595,382
"""baseline + interactions""",57,0.842,0.791,0.662,0.84,0.616,0.865,0.639,0.503,0.701,840,2751,523,428
"""extended""",17,0.856,0.801,0.694,0.842,0.63,0.877,0.66,0.536,0.723,880,2757,517,388
"""extended + interactions""",241,0.857,0.807,0.64,0.871,0.658,0.862,0.649,0.511,0.735,811,2853,421,457


Adding the eight further measured variables buys more than adding interaction
terms does: the 17-variable set beats the 9-variable set on every metric, while
the interaction terms mostly trade sensitivity for specificity at roughly
constant AUC.

## Gate G4

The legacy runs exported their training and test matrices alongside their
metrics, so this gate checks more than the published numbers:

1. **Split membership and order** — the full training (10,596 rows) and test
   (4,542 rows) matrices, cell by cell. This verifies that the split selected
   exactly the same participants in the same order, which is what makes the
   metrics comparable at all.
2. **Predicted probabilities** — the `preds` column of the exported test matrix.
3. **Metrics** — all nine plus the confusion matrix, against the stored JSON.

Note that `results_XGBoost.json` for three of the four runs records
`"description": "NHANES; ..."`. All four were trained on KNHANES; the label is
stale, carried over from a deleted notebook (`docs/audit.md` F10). The 4,542
test rows confirm the cohort.

In [3]:
import json

from src.data.io import repo_path
from src.validate import compare_matrices, report

RUN_FOLDERS = {
    "baseline": "20250310192814_V33",
    "baseline + interactions": "20250730193556_V37",
    "extended": "20250310192817_V34",
    "extended + interactions": "20250310192820_V35",
}
METRIC_KEYS = [
    "roc_auc", "accuracy", "sensitivity(recall)", "specificity", "PPV(precision)",
    "NPV", "f1_score", "Youdens_Index", "pr_auc",
]

results_root = Path(repo_path("reference_results_root"))
passed = True

for name, folder in RUN_FOLDERS.items():
    run_dir = results_root / folder
    training, testing = split_frames(runs[name])

    passed &= report(
        f"{name} [{folder}] training matrix",
        compare_matrices(training, pl.read_csv(run_dir / "training_data.csv")),
    )
    passed &= report(
        f"{name} [{folder}] test matrix and predictions",
        compare_matrices(testing, pl.read_csv(run_dir / "testing_data.csv")),
    )

    legacy = json.loads((run_dir / "results_XGBoost.json").read_text())
    metrics = compute_metrics(runs[name]["y_test"], runs[name]["preds"])
    differences = [
        f"{key}: {metrics[key]} != published {legacy[key]}"
        for key in METRIC_KEYS
        if metrics[key] != legacy[key]
    ]
    differences += [
        f"confusion_matrix.{key}: {metrics['confusion_matrix'][key]} != published {value}"
        for key, value in legacy["confusion_matrix"].items()
        if metrics["confusion_matrix"][key] != value
    ]
    passed &= report(
        f"{name} [{folder}] metrics and confusion matrix",
        {"passed": not differences, "differences": differences},
    )

print()
print("G4:", "PASS" if passed else "FAIL")

2026-09-13 21:56:16 [INFO] src.validate: PASS baseline [20250310192814_V33] training matrix


2026-09-13 21:56:16 [INFO] src.validate: PASS baseline [20250310192814_V33] test matrix and predictions


2026-09-13 21:56:16 [INFO] src.validate: PASS baseline [20250310192814_V33] metrics and confusion matrix


2026-09-13 21:56:16 [INFO] src.validate: PASS baseline + interactions [20250730193556_V37] training matrix


2026-09-13 21:56:16 [INFO] src.validate: PASS baseline + interactions [20250730193556_V37] test matrix and predictions


...

PASS  extended [20250310192817_V34] metrics and confusion matrix
PASS  extended + interactions [20250310192820_V35] training matrix


2026-09-13 21:56:16 [INFO] src.validate: PASS extended + interactions [20250310192820_V35] test matrix and predictions


PASS  extended + interactions [20250310192820_V35] test matrix and predictions


2026-09-13 21:56:17 [INFO] src.validate: PASS extended + interactions [20250310192820_V35] metrics and confusion matrix


PASS  extended + interactions [20250310192820_V35] metrics and confusion matrix

G4: PASS
